## Training Model

In [2]:
from google.colab import drive
import os

# Mount Drive with force remount to see recently uploaded files
drive.mount('/content/drive', force_remount=True)

# Define paths (Update ZIP_PATH if your file is in a subfolder)
ZIP_PATH = '/content/drive/MyDrive/processed.zip'
LOCAL_DATA_PATH = '/content/dataset'

if os.path.exists(ZIP_PATH):
    print(f" Found ZIP at {ZIP_PATH}")
    !unzip -o -q "{ZIP_PATH}" -d "{LOCAL_DATA_PATH}"
    print(" Unzip complete!")
else:
    print(" ZIP NOT FOUND! Run !find /content/drive/MyDrive -name 'processed.zip' to find it.")

Mounted at /content/drive
 Found ZIP at /content/drive/MyDrive/processed.zip
 Unzip complete!


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Assuming your frames are resized to 224x224
IMG_SIZE = (224, 224)
BATCH_SIZE = 8 # Keep small for video sequences

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    LOCAL_DATA_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    LOCAL_DATA_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 2556 images belonging to 1 classes.
Found 638 images belonging to 1 classes.


In [6]:
from tensorflow.keras import layers, models

def build_vision_guard_model():
    # Explicitly define input: (Frames, Height, Width, Channels)
    # We use 16 frames as per your previous error log
    inputs = layers.Input(shape=(16, 224, 224, 3))

    # Feature Extraction (CNN) wrapped in TimeDistributed
    x = layers.TimeDistributed(layers.Conv2D(32, (3, 3), activation='relu'))(inputs)
    x = layers.TimeDistributed(layers.MaxPooling2D((2, 2)))(x)
    x = layers.TimeDistributed(layers.Conv2D(64, (3, 3), activation='relu'))(x)
    x = layers.TimeDistributed(layers.GlobalAveragePooling2D())(x)

    # Sequence Processing (LSTM/GRU)
    x = layers.LSTM(64, return_sequences=False)(x)
    
    # Classification Head
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="VisionGuard_Model")
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

model = build_vision_guard_model()
model.summary()

Model: "VisionGuard_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 16, 224, 224,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 16, 222, 222,   │           896 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 111, 111,   │             0 │
│ (TimeDistributed)               │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 16, 109, 109,   │        18,496 │
│ (TimeDistributed)               │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 16, 64)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,641 (221.25 KB)

 Trainable params: 56,641 (221.25 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
EPOCHS = 10

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS
)

✅ Moved: train.csv -> /content/aimodel/dataset/
✅ Moved: val.csv -> /content/aimodel/dataset/
✅ Moved: test.csv -> /content/aimodel/dataset/
✅ Moved: metadata.csv -> /content/aimodel/dataset/

--- Current Contents of aimodel/dataset ---
['processed', 'train.csv', 'test.csv', 'metadata.csv', 'val.csv']


In [ ]:
import pandas as pd
import numpy as np
import cv2
import os
from tensorflow.keras import layers, models
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.utils import Sequence

# 1. Configuration
IMG_SIZE = (224, 224)
SEQ_LEN = 16 
BATCH_SIZE = 8

# 2. Define how to load the video frames
class VideoDataGenerator(Sequence):
    def __init__(self, csv_file, base_path, batch_size=8, seq_len=16):
        self.df = pd.read_csv(csv_file)
        self.base_path = base_path
        self.batch_size = batch_size
        self.seq_len = seq_len
        
    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        batch_df = self.df.iloc[index * self.batch_size : (index + 1) * self.batch_size]
        X = np.empty((self.batch_size, self.seq_len, *IMG_SIZE, 3))
        y = batch_df['label'].values
        for i, clip_rel_path in enumerate(batch_df['relative_path']):
            full_path = os.path.join(self.base_path, clip_rel_path)
            X[i] = self._load_frames(full_path)
        return X, y.astype('float32')

    def _load_frames(self, path):
        frames = []
        all_frames = sorted([f for f in os.listdir(path) if f.endswith(('.jpg', '.png'))])
        indices = np.linspace(0, len(all_frames) - 1, self.seq_len, dtype=int)
        for idx in indices:
            img = cv2.imread(os.path.join(path, all_frames[idx]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            frames.append(preprocess_input(img))
        return np.array(frames)

# 3. Build the Brain (CNN + LSTM)
base_cnn = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_cnn.trainable = False 

model = models.Sequential([
    layers.TimeDistributed(base_cnn, input_shape=(SEQ_LEN, 224, 224, 3)),
    layers.TimeDistributed(layers.GlobalAveragePooling2D()),
    layers.LSTM(64),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Initialize the Generators
# Make sure these CSV paths match where you moved your files!
# 4. Initialize the Generators
# We point to where the CSVs actually are and where the 'processed' folder is
TRAIN_CSV = "/content/dataset/train.csv"
VAL_CSV = "/content/dataset/val.csv"
DATA_ROOT = "/content/dataset/processed"

# Double check if these exist before running the generator
if os.path.exists(TRAIN_CSV) and os.path.exists(DATA_ROOT):
    train_gen = VideoDataGenerator(TRAIN_CSV, DATA_ROOT)
    val_gen = VideoDataGenerator(VAL_CSV, DATA_ROOT)
    print("✅ Generators initialized successfully!")
else:
    print("Paths are still wrong. Current /content contents:")
    !ls -R /content/dataset

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


FileNotFoundError: [Errno 2] No such file or directory: '/content/aimodel/dataset/train.csv'

In [13]:
# Run this to clean the paths in your generators
train_gen.df['relative_path'] = train_gen.df['relative_path'].str.replace('\\', '/', regex=False)
val_gen.df['relative_path'] = val_gen.df['relative_path'].str.replace('\\', '/', regex=False)

print("✅ CSV paths in memory have been converted to Linux format!")
print(f"Example path now: {train_gen.df['relative_path'].iloc[0]}")

✅ CSV paths in memory have been converted to Linux format!
Example path now: NonViolence/video_0


In [14]:
import os
os.makedirs('/content/aimodel/artifacts', exist_ok=True)
print("✅ 'artifacts' folder is ready. Your model has a place to live!")

✅ 'artifacts' folder is ready. Your model has a place to live!


In [15]:
import os
test_path = "/content/aimodel/dataset/processed/NonViolence/video_0"
if os.path.exists(test_path):
    print("🚀 Folder found! You are good to go.")
else:
    print("❌ Folder NOT found.")
    print("Checking what IS inside 'processed':", os.listdir("/content/aimodel/dataset/processed"))

🚀 Folder found! You are good to go.


In [16]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. Define where to save the results
checkpoint_path = '/content/aimodel/artifacts/best_model.h5'

# 2. Callbacks: Tools that watch the training
callbacks = [
    # Saves the model every time the AI gets smarter
    ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max'),
    # Stops training if the AI stops improving (saves time/money)
    EarlyStopping(patience=5, monitor='val_loss', restore_best_weights=True)
]

# 3. THE TRAINING LOOP
print("🚀 Starting Training...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,  # We'll start with 20; EarlyStopping will cut it short if needed
    callbacks=callbacks
)

print(f"✅ Training Finished! Best model saved to: {checkpoint_path}")

🚀 Starting Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5548 - loss: 0.7014

17/17 ━━━━━━━━━━━━━━━━━━━━ 226s 8s/step - accuracy: 0.5624 - loss: 0.6960 - val_accuracy: 0.9583 - val_loss: 0.3554
Epoch 2/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 102s 6s/step - accuracy: 0.9685 - loss: 0.2987 - val_accuracy: 0.9583 - val_loss: 0.1720
Epoch 3/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 98s 6s/step - accuracy: 1.0000 - loss: 0.1101 - val_accuracy: 0.9583 - val_loss: 0.0931
Epoch 4/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 97s 6s/step - accuracy: 1.0000 - loss: 0.0383 - val_accuracy: 0.9583 - val_loss: 0.0745
Epoch 5/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 99s 6s/step - accuracy: 1.0000 - loss: 0.0181 - val_accuracy: 0.9583 - val_loss: 0.1096
Epoch 6/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 99s 6s/step - accuracy: 1.0000 - loss: 0.0093 - val_accuracy: 0.9583 - val_loss: 0.1627
Epoch 7/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 1.0000 - loss: 0.0111

17/17 ━━━━━━━━━━━━━━━━━━━━ 97s 6s/step - accuracy: 1.0000 - loss: 0.0109 - val_accuracy: 1.0000 - val_loss: 0.0215
Epoch 8/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 98s 6s/step - accuracy: 1.0000 - loss: 0.0042 - val_accuracy: 0.9583 - val_loss: 0.0540
Epoch 9/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 104s 6s/step - accuracy: 1.0000 - loss: 0.0035 - val_accuracy: 0.9583 - val_loss: 0.0807
Epoch 10/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 97s 6s/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.9583 - val_loss: 0.1079
Epoch 11/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 96s 6s/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 0.9583 - val_loss: 0.1561
Epoch 12/20
17/17 ━━━━━━━━━━━━━━━━━━━━ 97s 6s/step - accuracy: 1.0000 - loss: 0.0014 - val_accuracy: 0.9583 - val_loss: 0.1834
✅ Training Finished! Best model saved to: /content/aimodel/artifacts/best_model.h5


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# This saves it into a folder named 'VisionGuard' in your Google Drive
import os
save_path = '/content/drive/MyDrive/VisionGuard/'
os.makedirs(save_path, exist_ok=True)

model.save(os.path.join(save_path, 'best_model.h5'))
print(f"✅ Saved to Google Drive at: {save_path}")

✅ Saved to Google Drive at: /content/drive/MyDrive/VisionGuard/
